In [1]:
import os, warnings
import numpy as np
import time

import numpy as np
import pandas as pd
import pickle
import os
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.insert(0,'../../uqmodels/abench')
sys.path.insert(0,'../../n5_uqmodels/')
sys.path.insert(0,'src/')
import abench 
import uqmodels

%load_ext autoreload
%autoreload 2

In [44]:
from abench.store.data_management import explore_csv_hierarchy
metadata = explore_csv_hierarchy('data',['dataset','set'])
if(False): # Delete data to 
    for n,sample in metadata.iterrows():
        path = sample['path']
        print(path)
        data = pd.read_csv(path)
        try:
            data = data.drop(['dataset', 'set','filename', 'seqId', 'length', 'cat_length', 'cat_edge'],axis=1)
        except:
            pass
        data.to_csv(path,index=False)

In [40]:
dataset = ['new_dataset10','new_dataset10_random_pos_offset_low'
           ,'new_dataset10_random_pos_offset_median','new_dataset10_random_pos_offset_high']

In [45]:
from abench.store.data_management import explore_csv_hierarchy,filter_metadata, augment_csvs_with_metadata
# metadata=filter_metadata(metadata,constraint_selection_list=[('dataset',dataset),('set',['new_dataset10_r1'])])
# metadata
metadata=filter_metadata(metadata,constraint_selection_list=[('dataset',['dataset1'])])
metadata

,dataset,set,filename,path
115,dataset1,set_4,batch_2.csv,data/dataset1/set_4/batch_2.csv
116,dataset1,set_4,batch_1.csv,data/dataset1/set_4/batch_1.csv
117,dataset1,set_4,batch_0.csv,data/dataset1/set_4/batch_0.csv
118,dataset1,set_4,batch_4.csv,data/dataset1/set_4/batch_4.csv
119,dataset1,set_4,batch_3.csv,data/dataset1/set_4/batch_3.csv
120,dataset1,set_3,batch_2.csv,data/dataset1/set_3/batch_2.csv
121,dataset1,set_3,batch_1.csv,data/dataset1/set_3/batch_1.csv
122,dataset1,set_3,batch_0.csv,data/dataset1/set_3/batch_0.csv
123,dataset1,set_3,batch_4.csv,data/dataset1/set_3/batch_4.csv
124,dataset1,set_3,batch_3.csv,data/dataset1/set_3/batch_3.csv


In [35]:
# Rajout des informations contextuelle : Metadata + Descripteur + Categorization des descripteurs

from src.descriptor import macro_id,calculate_length
from functools import partial

descriptors = {
    "macro_seqId":macro_id,
    "length":calculate_length}         

# -----------------------------------------------------------------------------
# Quantify descrptor
# -----------------------------------------------------------------------------
enrich_params = {'macro_descriptors':{"seqId":macro_id},
                 'group_descriptors':{"length":partial(calculate_length,x_col="positionX",y_col="positionY",time_col="ts")},
                 'quantize':{ "length":('quantile',[0, 0.50, 0.75, 0.90, 1.0]),
                           "edge":('pattern',{'-60546309': 1, '60546306': 2, '118438448': 3, '1090414517': 4, '-172341587': 5, '60546304': 6, '60546309': 7, 
                                              '1090414515': 8, '-60546312': 9, '1090414518': 10, '60546290': 11, '172341587': 12, '60546311': 13, '60546303': 14, 
                                              '-118438448': 15, '1090414516': 16, '60546312': 17})},
                 'Id_group':"seqId",
                 'time_col':"ts"}

from abench.store.data_management import explore_csv_hierarchy,filter_metadata, augment_csvs_with_metadata
metadata = explore_csv_hierarchy('data',['dataset','set'])
metadata=filter_metadata(metadata,constraint_selection_list=[('dataset',dataset)],
                         constraint_rejection_list=[('set',dataset)])
metadata
augment_csvs_with_metadata(metadata_df=metadata,columns_to_add=['dataset', 'set', 'filename'],enrich_params=enrich_params)
# augment_csvs_with_metadata(metadata_df=metadata,columns_to_add=['dataset', 'set', 'filename'])

In [ ]:
# Creation d'un data set perturbés.
from functools import partial
from pathlib import Path
from src.perturbator import gaussian_noise_perturbator, apply_perturbations
from abench.store.data_management import explore_csv_hierarchy,filter_metadata

In [ ]:

metadata = explore_csv_hierarchy('data',['dataset','set'])
metadata=filter_metadata(metadata,constraint_selection_list=[('dataset',[dataset])])

dict_perturbators={"noisy":{"noisy":partial(gaussian_noise_perturbator,x_col='positionX', y_col='positionY',sigma=0.5)}}
                  
for key,perturbators in dict_perturbators.items():
    for n,sample in metadata.iterrows():
        path = sample['path']
        df = pd.read_csv(path)
        df_perturbation = apply_perturbations(df,macro_perturbation=perturbators,Id_col="seqId",time_col="ts")
        path = path.replace(dataset,dataset+'_'+key)
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        df_perturbation.to_csv(path,index=False)

In [ ]:
metadata = explore_csv_hierarchy('data',['dataset','set'])
constraint_selection_list=[('dataset',[dataset]),('set',['set_1','set_0'])]
metadata=filter_metadata(metadata,constraint_selection_list=constraint_selection_list)
for col, allowed_values in constraint_selection_list:
    print(col,allowed_values)
metadata

In [ ]:
# Creation d'un data set perturbés.
from functools import partial
from pathlib import Path
from src.perturbator import gaussian_noise_perturbator, apply_perturbations
from abench.store.data_management import explore_csv_hierarchy,filter_metadata
metadata = explore_csv_hierarchy('data',['dataset','set'])
metadata=filter_metadata(metadata,constraint_selection_list=[('dataset',[dataset])])
dict_perturbators={"noisy_eps":{"noisy_eps":partial(gaussian_noise_perturbator,x_col='positionX', y_col='positionY',sigma=0.01)}}
for key,perturbators in dict_perturbators.items():
    for n,sample in metadata.iterrows():
        path = sample['path']
        df = pd.read_csv(path)
        df_perturbation = apply_perturbations(df,macro_perturbation=perturbators,Id_col="seqId",time_col="ts")
        path = path.replace(dataset,dataset+'_'+key)
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        df_perturbation.to_csv(path,index=False)

In [ ]:
# Creation d'un data set perturbés.
from functools import partial
from pathlib import Path
from src.perturbator import gaussian_noise_perturbator, piecewise_linear_perturbator, apply_perturbations
from abench.store.data_management import explore_csv_hierarchy,filter_metadata
metadata = explore_csv_hierarchy('data',['dataset','set'])
metadata=filter_metadata(metadata,constraint_selection_list=[('dataset',[dataset])])
dict_perturbators = {"piecewise": partial(piecewise_linear_perturbator,x_col="positionX",y_col="positionY",anchor_indices=[0, 16, 36, 50])}
for key,perturbators in dict_perturbators.items():
    for n,sample in metadata.iterrows():
        path = sample['path']
        df = pd.read_csv(path)
        df_perturbation = apply_perturbations(df,group_perturbation={key:perturbators},Id_col="seqId",time_col="ts")
        path = path.replace(dataset,dataset+'_'+key)
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        df_perturbation.to_csv(path,index=False)

In [ ]:
from abench.store.data_management import explore_csv_hierarchy,filter_metadata
from functools import partial
from pathlib import Path
from src.perturbator import gaussian_noise_perturbator, piecewise_linear_perturbator, apply_perturbations
from abench.store.data_management import explore_csv_hierarchy,filter_metadata
metadata = explore_csv_hierarchy('data',['dataset','set'])
metadata=filter_metadata(metadata,constraint_selection_list=[('dataset',[dataset])])
dict_perturbators = {"piecewise_on_cat2": partial(piecewise_linear_perturbator,x_col="positionX",y_col="positionY",anchor_indices=[0, 16, 36, 50],check_col='cat_length',allowed_values=[2])}
for key,perturbators in dict_perturbators.items():
    for n,sample in metadata.iterrows():
        path = sample['path']
        df = pd.read_csv(path)
        df_perturbation = apply_perturbations(df,group_perturbation={key:perturbators},Id_col="seqId",time_col="ts")
        path = path.replace(dataset,dataset+'_'+key)
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        df_perturbation.to_csv(path,index=False)

# constant offset attack

In [2]:
from abench.store.data_management import explore_csv_hierarchy,filter_metadata
metadata = explore_csv_hierarchy('data',['dataset','set'])
metadata=filter_metadata(metadata,constraint_selection_list=[('dataset',['new_dataset10']),('set',['new_dataset10'])])
metadata

,dataset,set,filename,path
64,new_dataset10,new_dataset10,training_0_999_20261002_14:36:30.csv,data/new_dataset10/new_dataset10/training_0_99...


In [18]:
# Creation d'un data set perturbés.
from functools import partial
from pathlib import Path
from src.perturbator import gaussian_noise_perturbator, piecewise_linear_perturbator, apply_perturbations, random_pos_offset, const_pos_offset
from abench.store.data_management import explore_csv_hierarchy,filter_metadata
import yaml
metadata = explore_csv_hierarchy('data',['dataset','set'])
metadata=filter_metadata(metadata,constraint_selection_list=[('dataset',['new_dataset10']),('set',['new_dataset10'])])

config_yaml_path = '/home/ctm/Documents/ML/MLTest/symaps_data_analyze/benchmark/config/config_attack.yaml'
with open(config_yaml_path) as f:
    config_attack = yaml.safe_load(f)    

with open(config_attack['template']) as f:
    config_template = yaml.safe_load(f)  

    
cfg_params=pd.concat([pd.DataFrame.from_dict(config_attack, orient='index'),pd.DataFrame.from_dict(config_template, orient='index')])
dict_perturbators = {"random_pos_offset_low": partial(random_pos_offset,config_yaml=config_attack)}
for key,perturbators in dict_perturbators.items():
    for n,sample in metadata.iterrows():
        path = sample['path']
        df = pd.read_csv(path)
        df_perturbation = apply_perturbations(df,group_perturbation={key:perturbators},Id_col="Id",time_col="ts")
        path = path.replace('new_dataset10','new_dataset10'+'_'+key)
        cfg_path = path.replace('csv','log')
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        cfg_params.to_csv(cfg_path,sep=':', mode='w',header=False)
        df_perturbation.to_csv(path,index=False)

print("finish")

finish


In [43]:
# Creation d'un data set perturbés.
from functools import partial
from pathlib import Path
from src.perturbator import gaussian_noise_perturbator, piecewise_linear_perturbator, apply_perturbations, random_pos_offset, const_pos_offset
from abench.store.data_management import explore_csv_hierarchy,filter_metadata
import yaml
metadata = explore_csv_hierarchy('data',['dataset','set'])
metadata=filter_metadata(metadata,constraint_selection_list=[('dataset',['new_dataset10']),('set',['new_dataset10'])])

config_yaml_path = '/home/ctm/Documents/ML/MLTest/symaps_data_analyze/benchmark/config/config_attack.yaml'
with open(config_yaml_path) as f:
    config_attack = yaml.safe_load(f)    

with open(config_attack['template']) as f:
    config_template = yaml.safe_load(f)  

    
cfg_params=pd.concat([pd.DataFrame.from_dict(config_attack, orient='index'),pd.DataFrame.from_dict(config_template, orient='index')])
dict_perturbators = {"const_pos_offset": partial(const_pos_offset,config_yaml=config_attack)}
for key,perturbators in dict_perturbators.items():
    for n,sample in metadata.iterrows():
        path = sample['path']
        df = pd.read_csv(path)
        df_perturbation = apply_perturbations(df,group_perturbation={key:perturbators},Id_col="Id",time_col="ts")
        path = path.replace('new_dataset10','new_dataset10'+'_'+key)
        cfg_path = path.replace('csv','log')
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        cfg_params.to_csv(cfg_path,sep=':', mode='w',header=False)
        df_perturbation.to_csv(path,index=False)

In [17]:
df = pd.DataFrame(
    [[1, 2], [4, 5], [7, 8]],
    columns=["max_speed", "shield"],
)
qq = pd.DataFrame([1,2,4],columns=['shield'])
dd = pd.DataFrame([2,3,4])
df[1:50]['shield'] = df[1:50]['shield']  + dd[1:50][0]
df

,max_speed,shield
0,1,2
1,4,5
2,7,8


In [ ]:
from src.visu import plot_trajet
df = pd.read_csv('data/dataset10/set_1/batch_0.csv')
mask_1 = (df['Id']==(df['Id'].values[0]))
mask_2 = (df['Id']==(df['Id'].values[458]))
mask_3 = (df['Id']==(df['Id'].values[1256]))
mask_4 = (df['Id']==(df['Id'].values[230]))


df_traj = [[df[mask_1][['positionX','positionY']].values,
            df[mask_2][['positionX','positionY']].values],
           [df[mask_3][['positionX','positionY']].values,
            df[mask_4][['positionX','positionY']].values]]

df = pd.read_csv('data/dataset10_noisy/set_1/batch_0.csv')
df_traj_noisy = [[df[mask_1][['positionX','positionY']].values,
                  df[mask_2][['positionX','positionY']].values],
                 [df[mask_3][['positionX','positionY']].values,
                  df[mask_4][['positionX','positionY']].values]]

df = pd.read_csv('data/dataset10_noisy_eps/set_1/batch_0.csv')
df_traj_noisy_eps = [[df[mask_1][['positionX','positionY']].values,
                      df[mask_2][['positionX','positionY']].values], 
                     [df[mask_3][['positionX','positionY']].values,
                      df[mask_4][['positionX','positionY']].values]]

df = pd.read_csv('data/dataset10_piecewise/set_1/batch_0.csv')
df_traj_piecewise = [[df[mask_1][['positionX','positionY']].values,
                      df[mask_2][['positionX','positionY']].values], 
                     [df[mask_3][['positionX','positionY']].values,
                      df[mask_4][['positionX','positionY']].values]]

df = pd.read_csv('data/dataset10_piecewise_on_cat2/set_1/batch_0.csv')
df_traj_piecewise_on_cat2 = [[df[mask_1][['positionX','positionY']].values,
                              df[mask_2][['positionX','positionY']].values],
                             [df[mask_3][['positionX','positionY']].values,
                              df[mask_4][['positionX','positionY']].values]]


import matplotlib.pyplot as plt
fig,ax = plt.subplots(nrows=2, ncols=2,figsize=(20,20))
for i in range(2):
    for j in range(2):
        plot_trajet(ax=ax[i][j],y=df_traj[i][j],colors= ("green", "red"))
        plot_trajet(ax=ax[i][j],y=df_traj_noisy[i][j],colors= ("blue", "red"))
        plot_trajet(ax=ax[i][j],y=df_traj_piecewise[i][j],colors= ("red", "red"))
        plot_trajet(ax=ax[i][j],y=df_traj_noisy_eps[i][j],colors= ("yellow", "yellow"))
        plot_trajet(ax=ax[i][j],y=df_traj_piecewise_on_cat2[i][j],colors= ("purple", "purple"))
